# 02 Frames and aggregation

Per-frame features, their time axis, plots, and the ways AFX reduces frames to one vector per clip. Plots need `pip install "afx-audio[plot]"`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import afx_audio as afx
from afx_audio.utils.visualization import plot_mfcc, plot_spectrogram
import demo_data

SR = 16000
FRAME, HOP = 1024, 256

## A test signal

A dataset clip if one is configured, otherwise a harmonic tone followed by a sweep. Half a second of digital silence is appended, where pitch-based features are undefined.

In [ ]:
files = demo_data.find_audio(max_files=1)
if files:
    clip, _ = afx.load_audio(files[0], sr=SR, duration=3.0)
else:
    clip = np.concatenate([demo_data.synth_clip('tone', 1.5, SR, seed=3),
                           demo_data.synth_clip('chirp', 1.5, SR, seed=4)])
signal = np.concatenate([clip, np.zeros(SR // 2, dtype=np.float32)])
print(f'{signal.size / SR:.2f} s at {SR} Hz')

## Per-frame arrays

`preserve_shape=True` skips aggregation. One-value features come back as `(n_frames,)`, multi-value ones as `(n_values, n_frames)`. `return_metadata=True` adds each feature's frame times (frame centres, in seconds).

In [ ]:
frame_cfg = {'frame_size': FRAME, 'hop_length': HOP}
config = {
    'sample_rate': SR,
    'features': {
        'melspectrogram': {**frame_cfg, 'n_mels': 64, 'center': False},
        'mfcc': {**frame_cfg, 'n_mfcc': 13},
        'spectral_centroid': frame_cfg,
        'rms_energy': frame_cfg,
        'pitch': frame_cfg,
        'hnr': frame_cfg,
    },
    'preserve_shape': True,
}
out = afx.extract_all_features(signal, SR, config, return_metadata=True)
frames, meta = out['features'], out['metadata']
for name, value in frames.items():
    print(f'{name:18s} {str(value.shape):12s} first frame at {meta[name]["times"][0]:.3f} s')

Features share one time axis when `frame_size`, `hop_length` and `center` agree, so they can be stacked column by column. `melspectrogram`, `chroma_stft`, `cqt`, `tonnetz`, `spectral_contrast` and `spectral_flux` default to `center=True` (librosa's convention, frames centred on `t = k * hop`) and the other framed features to `False`, which is why the config above sets `center` for the mel spectrogram.

In [ ]:
times = meta['mfcc']['times']
assert all(np.allclose(m['times'], times) for m in meta.values())
print(f'{times.size} frames, hop {np.diff(times)[0] * 1000:.0f} ms')

## Plots

`afx_audio.utils.visualization` has `plot_spectrogram` for an STFT power spectrogram (rows are linear frequency bins), `plot_mfcc` and `plot_feature_distribution`. Each accepts an `ax` to draw into. The STFT comes from `afx_audio.methods.stft`, the transform the spectral extractors use.

In [ ]:
from afx_audio.methods.stft import power_spectrogram, stft
from afx_audio.utils.visualization import plot_feature_distribution

fig, axes = plt.subplots(3, 1, figsize=(10, 9))
power = power_spectrogram(stft(signal, frame_size=FRAME, hop_length=HOP, center=False))
plot_spectrogram(power, SR, hop_length=HOP, title='STFT power', ax=axes[0])
plot_mfcc(frames['mfcc'], SR, hop_length=HOP, ax=axes[1])

ax = axes[2]
ax.plot(times, frames['spectral_centroid'], label='spectral centroid (Hz)')
ax.plot(times, frames['pitch'], label='pitch (Hz, 0 = unvoiced)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Hz')
ax2 = ax.twinx()
ax2.plot(times, frames['hnr'], color='k', lw=1, label='HNR (dB)')
ax2.set_ylabel('HNR (dB)')
ax.legend(loc='upper left')
ax2.legend(loc='upper right')
ax.set_title('Frame-level descriptors; HNR has gaps where it is undefined')
fig.tight_layout()

hnr_frames = frames['hnr']
plot_feature_distribution(hnr_frames[~np.isnan(hnr_frames)], title='HNR over defined frames (dB)')

## Undefined frames are NaN

HNR is undefined on silent frames, so those frames are NaN rather than an arbitrary number. Every aggregation skips NaN frames; a statistic is NaN only when all frames are.

In [ ]:
hnr = frames['hnr']
print(f'HNR: {np.isnan(hnr).sum()} of {hnr.size} frames are NaN')
print('np.mean          :', np.mean(hnr))
print('aggregated mean  :', afx.aggregate_features({'hnr': hnr}, method='mean')['hnr'])

silent = afx.extract_all_features(np.zeros(SR, dtype=np.float32), SR, {'features': {'hnr': frame_cfg}})
print('all-silent clip  :', silent['hnr'])

A clip-level NaN reaches the feature matrix as a missing value. In scikit-learn put a `SimpleImputer` before the scaler (see `05_sklearn_and_selection.ipynb`).

## Aggregation methods

`aggregate_features` (or the `aggregation` config key) reduces the time axis.

| method | output per feature |
|---|---|
| `mean`, `std`, `skew`, `kurtosis` | one statistic per row |
| `summary` | `<name>_mean`, `<name>_std` |
| `advanced_summary` | mean, std, skew, kurtosis, each as its own key |
| `sliding_window` | windowed statistic, `(rows, n_windows)` |
| `stack` | every frame of every feature in one 1-D array |

In [ ]:
per_frame = {k: frames[k] for k in ('mfcc', 'spectral_centroid', 'hnr')}
for method in ('mean', 'std', 'skew', 'kurtosis', 'summary', 'advanced_summary'):
    agg = afx.aggregate_features(per_frame, method=method)
    names, vector = afx.flatten_features(agg)
    print(f'{method:17s} {vector.size:4d} values  keys: {list(agg)}')

stacked = afx.aggregate_features(per_frame, method='stack')
print(f'{"stack":17s} {stacked.size:4d} values (13 + 1 + 1 rows x {times.size} frames)')

`sliding_window` pools `window_size` frames every `hop_length` frames (both counted in frames), giving a coarser time series.

In [ ]:
windows = afx.aggregate_features(per_frame, method='sliding_window', window_size=16, hop_length=8,
                                 sliding_agg='mean', flatten=False)
for name, value in windows.items():
    print(f'{name:18s} {value.shape}')

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(times, frames['spectral_centroid'], alpha=0.4, label='per frame')
window_times = times[0] + (np.arange(windows['spectral_centroid'].shape[-1]) * 8 + 7.5) * HOP / SR
ax.plot(window_times, windows['spectral_centroid'].ravel(), 'o-', label='16-frame mean, hop 8')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Hz')
ax.legend()
fig.tight_layout()

The same through a config: `aggregation` picks the method and `aggregation_params` carries its arguments.

In [ ]:
config_sw = {**config, 'preserve_shape': False, 'aggregation': 'sliding_window',
             'aggregation_params': {'window_size': 16, 'hop_length': 8}, 'flatten': False}
print({k: v.shape for k, v in afx.extract_all_features(signal, SR, config_sw).items()})

## Normalization

`normalize='zscore'` or `'minmax'` rescales each clip by its own statistics, which makes values incomparable across clips. For a dataset, compute statistics on the training set and pass them as `norm_stats`, or scale in the model pipeline.

In [ ]:
agg = afx.aggregate_features(per_frame, method='mean')
stats = {'mean': {k: 0.0 for k in agg}, 'std': {'mfcc': 50.0, 'spectral_centroid': 1000.0, 'hnr': 10.0}}
scaled = afx.aggregate_features(per_frame, method='mean', normalize='zscore', norm_stats=stats)
print({k: np.round(v[:3], 3) for k, v in scaled.items()})